In [ ]:
# ============================================================
# LICENSE PLATE DETECTION + OCR
# YOLO11n + License Plate YOLO + EasyOCR
# ============================================================

import cv2
import os
import re
import numpy as np
import easyocr

from ultralytics import YOLO
from IPython.display import Video, display


# ============================================================
# 1. PATHS
# ============================================================

VIDEO_PATH = (
    r"C:\Users\PC-LAB1\Desktop\New folder\lic"
    r"\License_Plate_Detection_Test.mp4"
)

CAR_MODEL_PATH = (
    r"C:\Users\PC-LAB1\Desktop\shok"
    r"\notebooks\Day_09\speed\yolo11n.pt"
)

LICENSE_MODEL_PATH = (
    r"C:\Users\PC-LAB1\Desktop\shok"
    r"\runs\detect\train-2\weights\best.pt"
)

OUTPUT_VIDEO = (
    r"C:\Users\PC-LAB1\Desktop\New folder\lic"
    r"\License_Plate_Detection_OCR.mp4"
)


# ============================================================
# 2. CHECK FILES
# ============================================================

for path, name in [
    (VIDEO_PATH, "Video"),
    (CAR_MODEL_PATH, "YOLO11n car model"),
    (LICENSE_MODEL_PATH, "License plate model")
]:

    if not os.path.exists(path):

        raise FileNotFoundError(
            f"{name} not found:\n{path}"
        )


# ============================================================
# 3. LOAD MODELS
# ============================================================

print("Loading YOLO11n...")

car_model = YOLO(
    CAR_MODEL_PATH
)

print("YOLO11n loaded.")


print()
print("Loading license plate model...")

plate_model = YOLO(
    LICENSE_MODEL_PATH
)

print("License plate model loaded.")


# ============================================================
# 4. LOAD EASYOCR
# ============================================================

print()
print("Loading EasyOCR...")

reader = easyocr.Reader(
    ['en'],
    gpu=False
)

print("EasyOCR loaded.")


# ============================================================
# 5. OPEN VIDEO
# ============================================================

cap = cv2.VideoCapture(
    VIDEO_PATH
)

if not cap.isOpened():

    raise RuntimeError(
        "Could not open video."
    )


FPS = cap.get(
    cv2.CAP_PROP_FPS
)

FRAME_COUNT = int(
    cap.get(
        cv2.CAP_PROP_FRAME_COUNT
    )
)

WIDTH = int(
    cap.get(
        cv2.CAP_PROP_FRAME_WIDTH
    )
)

HEIGHT = int(
    cap.get(
        cv2.CAP_PROP_FRAME_HEIGHT
    )
)


print()
print("========================================")
print("VIDEO INFORMATION")
print("========================================")

print(
    f"Resolution : {WIDTH} x {HEIGHT}"
)

print(
    f"FPS        : {FPS:.2f}"
)

print(
    f"Frames     : {FRAME_COUNT}"
)

print(
    f"Duration   : {FRAME_COUNT / FPS:.2f} seconds"
)


# ============================================================
# 6. OUTPUT VIDEO
# ============================================================

os.makedirs(
    os.path.dirname(OUTPUT_VIDEO),
    exist_ok=True
)


fourcc = cv2.VideoWriter_fourcc(
    *"mp4v"
)


writer = cv2.VideoWriter(
    OUTPUT_VIDEO,
    fourcc,
    FPS,
    (WIDTH, HEIGHT)
)


if not writer.isOpened():

    raise RuntimeError(
        "Could not create output video."
    )


# ============================================================
# 7. SETTINGS
# ============================================================

CAR_CONFIDENCE = 0.25

PLATE_CONFIDENCE = 0.20

OCR_CONFIDENCE = 0.20


# COCO vehicle classes
#
# 2 = car
# 3 = motorcycle
# 5 = bus
# 7 = truck

CAR_CLASSES = [
    2,
    3,
    5,
    7
]


# ============================================================
# 8. OCR CLEANING
# ============================================================

def clean_text(text):

    text = text.upper()

    # Remove everything except
    # letters and numbers

    text = re.sub(
        r'[^A-Z0-9]',
        '',
        text
    )

    return text


# ============================================================
# 9. OCR FUNCTION
# ============================================================

def read_plate(crop):

    if crop is None:
        return "", 0.0


    if crop.size == 0:
        return "", 0.0


    # --------------------------------------------------------
    # Upscale plate
    # --------------------------------------------------------

    crop = cv2.resize(
        crop,
        None,
        fx=3,
        fy=3,
        interpolation=cv2.INTER_CUBIC
    )


    # --------------------------------------------------------
    # Improve contrast
    # --------------------------------------------------------

    gray = cv2.cvtColor(
        crop,
        cv2.COLOR_BGR2GRAY
    )


    gray = cv2.bilateralFilter(
        gray,
        7,
        50,
        50
    )


    # --------------------------------------------------------
    # OCR
    # --------------------------------------------------------

    results = reader.readtext(
        gray,
        detail=1,
        paragraph=False,
        allowlist="ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789"
    )


    if not results:

        return "", 0.0


    texts = []

    confidences = []


    for result in results:

        if len(result) < 3:
            continue


        text = result[1]

        confidence = float(
            result[2]
        )


        text = clean_text(
            text
        )


        if len(text) == 0:
            continue


        if confidence < OCR_CONFIDENCE:
            continue


        texts.append(
            text
        )

        confidences.append(
            confidence
        )


    if not texts:

        return "", 0.0


    final_text = "".join(
        texts
    )


    final_confidence = float(
        np.mean(
            confidences
        )
    )


    return (
        final_text,
        final_confidence
    )


# ============================================================
# 10. TRACKING STORAGE
# ============================================================

plate_results = {}

frame_number = 0

plates_detected = 0

ocr_successes = 0


# ============================================================
# 11. PROCESS VIDEO
# ============================================================

print()
print("========================================")
print("PROCESSING VIDEO")
print("========================================")


while True:

    ret, frame = cap.read()


    if not ret:
        break


    frame_number += 1


    output = frame.copy()


    # ========================================================
    # CAR DETECTION
    # ========================================================

    car_results = car_model(
        frame,
        classes=CAR_CLASSES,
        conf=CAR_CONFIDENCE,
        verbose=False
    )


    # ========================================================
    # DRAW CAR BOXES
    # ========================================================

    if (
        len(car_results) > 0
        and
        car_results[0].boxes is not None
    ):

        boxes = car_results[0].boxes


        xyxy = (
            boxes.xyxy
            .cpu()
            .numpy()
        )


        classes = (
            boxes.cls
            .cpu()
            .numpy()
            .astype(int)
        )


        confidences = (
            boxes.conf
            .cpu()
            .numpy()
        )


        for (
            box,
            cls,
            confidence
        ) in zip(
            xyxy,
            classes,
            confidences
        ):

            x1, y1, x2, y2 = map(
                int,
                box
            )


            # Green car box

            cv2.rectangle(
                output,
                (x1, y1),
                (x2, y2),
                (0, 255, 0),
                2
            )


            # ------------------------------------------------
            # Vehicle name
            # ------------------------------------------------

            names = {
                2: "car",
                3: "motorcycle",
                5: "bus",
                7: "truck"
            }


            vehicle_name = names.get(
                cls,
                "vehicle"
            )


            label = (
                f"{vehicle_name} "
                f"{confidence:.2f}"
            )


            cv2.putText(
                output,
                label,
                (x1, max(y1 - 8, 20)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.55,
                (0, 255, 0),
                2,
                cv2.LINE_AA
            )


    # ========================================================
    # LICENSE PLATE DETECTION
    # ========================================================

    plate_results_frame = plate_model(
        frame,
        conf=PLATE_CONFIDENCE,
        verbose=False
    )


    if (
        len(plate_results_frame) > 0
        and
        plate_results_frame[0].boxes is not None
    ):

        plate_boxes = (
            plate_results_frame[0]
            .boxes
        )


        plate_xyxy = (
            plate_boxes.xyxy
            .cpu()
            .numpy()
        )


        plate_confidences = (
            plate_boxes.conf
            .cpu()
            .numpy()
        )


        for (
            plate_box,
            plate_confidence
        ) in zip(
            plate_xyxy,
            plate_confidences
        ):

            plates_detected += 1


            px1, py1, px2, py2 = map(
                int,
                plate_box
            )


            # Keep coordinates inside frame

            px1 = max(
                0,
                px1
            )

            py1 = max(
                0,
                py1
            )

            px2 = min(
                WIDTH,
                px2
            )

            py2 = min(
                HEIGHT,
                py2
            )


            if px2 <= px1 or py2 <= py1:

                continue


            # ------------------------------------------------
            # Crop plate
            # ------------------------------------------------

            plate_crop = frame[
                py1:py2,
                px1:px2
            ]


            # ------------------------------------------------
            # OCR
            # ------------------------------------------------

            text, ocr_confidence = read_plate(
                plate_crop
            )


            if text:

                ocr_successes += 1


                plate_results[
                    text
                ] = plate_results.get(
                    text,
                    0
                ) + 1


            # ------------------------------------------------
            # Plate bounding box
            # ------------------------------------------------

            cv2.rectangle(
                output,
                (px1, py1),
                (px2, py2),
                (0, 0, 255),
                2
            )


            # ------------------------------------------------
            # Plate label
            # ------------------------------------------------

            if text:

                label = (
                    f"{text} "
                    f"{ocr_confidence:.2f}"
                )

            else:

                label = (
                    "plate "
                    f"{plate_confidence:.2f}"
                )


            text_y = max(
                py1 - 8,
                20
            )


            # Background

            (
                tw,
                th
            ), baseline = cv2.getTextSize(
                label,
                cv2.FONT_HERSHEY_SIMPLEX,
                0.55,
                2
            )


            cv2.rectangle(
                output,
                (
                    px1,
                    text_y - th - baseline
                ),
                (
                    px1 + tw,
                    text_y + baseline
                ),
                (0, 0, 0),
                -1
            )


            cv2.putText(
                output,
                label,
                (
                    px1,
                    text_y
                ),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.55,
                (255, 255, 255),
                2,
                cv2.LINE_AA
            )


    # ========================================================
    # INFORMATION PANEL
    # ========================================================

    cv2.rectangle(
        output,
        (5, 5),
        (350, 85),
        (0, 0, 0),
        -1
    )


    cv2.putText(
        output,
        f"Frame: {frame_number}/{FRAME_COUNT}",
        (12, 27),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.55,
        (255, 255, 255),
        1,
        cv2.LINE_AA
    )


    cv2.putText(
        output,
        f"Plates detected: {plates_detected}",
        (12, 50),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.55,
        (255, 255, 255),
        1,
        cv2.LINE_AA
    )


    cv2.putText(
        output,
        f"OCR reads: {ocr_successes}",
        (12, 73),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.55,
        (255, 255, 255),
        1,
        cv2.LINE_AA
    )


    # ========================================================
    # WRITE FRAME
    # ========================================================

    writer.write(
        output
    )


    # ========================================================
    # PROGRESS
    # ========================================================

    if (
        frame_number % 10 == 0
        or
        frame_number == FRAME_COUNT
    ):

        print(
            f"Processed "
            f"{frame_number}/{FRAME_COUNT}",
            end="\r"
        )


# ============================================================
# CLEANUP
# ============================================================

cap.release()

writer.release()


# ============================================================
# RESULTS
# ============================================================

print()
print()

print("========================================")
print("PROCESSING COMPLETE")
print("========================================")

print(
    f"Frames processed: "
    f"{frame_number}"
)

print(
    f"License plates detected: "
    f"{plates_detected}"
)

print(
    f"Successful OCR reads: "
    f"{ocr_successes}"
)


print()
print("Detected plate texts:")
print("----------------------------------------")


if plate_results:

    # Sort by number of appearances

    sorted_results = sorted(
        plate_results.items(),
        key=lambda x: x[1],
        reverse=True
    )


    for text, count in sorted_results:

        print(
            f"{text:20s} "
            f"{count} detections"
        )

else:

    print(
        "No readable plate text found."
    )


# ============================================================
# OUTPUT
# ============================================================

print()
print("Output video:")
print(
    OUTPUT_VIDEO
)


# ============================================================
# DISPLAY VIDEO
# ============================================================

print()
print("Displaying result...")

display(
    Video(
        OUTPUT_VIDEO,
        embed=True,
        width=960
    )
)